# Heart Disease Classification — Exploratory Data Analysis

**Dataset:** UCI Cleveland Heart Disease  
**Objective:** Explore the data and understand factors associated with heart disease.

The original target ranges from 0 to 4. For this project's binary classification task, target 0 means no disease and values greater than 0 mean disease present.

This analysis is educational and is not intended for clinical diagnosis.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"
df = pd.read_csv(DATA_PATH, na_values="?")

print("Dataset shape:", df.shape)
display(df.head())


## 1. Dataset structure and summary

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nSummary statistics:")
display(df.describe().T)


## 2. Missing values and data quality

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Missing values by column:")
display(missing.to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())

df = df.dropna().copy()
df["target"] = (df["target"] > 0).astype(int)

print("Shape after dropping missing rows:", df.shape)
print("\nBinary target counts:")
display(df["target"].value_counts().sort_index().rename(
    index={0: "No disease", 1: "Disease present"}
).to_frame("count"))


## 3. Target distribution

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="target")
plt.title("Binary Heart Disease Target Distribution")
plt.xlabel("Target (0 = no disease, 1 = disease present)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


## 4. Feature distributions

In [ ]:
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

df[numeric_features].hist(figsize=(12, 8), bins=20, edgecolor="black")
plt.suptitle("Distributions of Numeric Features", y=1.02)
plt.tight_layout()
plt.show()


## 5. Numeric features by target

In [ ]:
for feature in numeric_features:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df, x="target", y=feature)
    plt.title(f"{feature} by Heart Disease Target")
    plt.xlabel("Target (0 = no disease, 1 = disease present)")
    plt.tight_layout()
    plt.show()


## 6. Correlation analysis

In [ ]:
plt.figure(figsize=(12, 9))
correlation = df.corr(numeric_only=True)
sns.heatmap(correlation, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

print("Features most correlated with the binary target:")
display(correlation["target"].drop("target").sort_values(
    key=abs, ascending=False
).to_frame("correlation_with_target"))


## 7. Categorical feature comparisons

In [ ]:
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

for feature in categorical_features:
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x=feature, hue="target")
    plt.title(f"{feature} category counts by target")
    plt.xlabel(feature)
    plt.ylabel("Number of records")
    plt.legend(title="Target", labels=["No disease", "Disease present"])
    plt.tight_layout()
    plt.show()


## 8. Key observations and conclusion

In [ ]:
print(f"Records available after cleaning: {len(df)}")
print(f"Input features: {df.shape[1] - 1}")
print(f"Records with disease label 1: {(df['target'] == 1).sum()}")
print(f"Records with no-disease label 0: {(df['target'] == 0).sum()}")

print("\nAbsolute correlations with target (exploratory only):")
display(
    df.corr(numeric_only=True)["target"]
    .drop("target")
    .abs()
    .sort_values(ascending=False)
    .head(5)
    .to_frame("absolute_correlation")
)

print("\nConclusion:")
print("- The dataset contains clinical measurements and categorical attributes.")
print("- Missing values were removed here to match the project's data preparation step.")
print("- The original target was converted to binary labels for this classification project.")
print("- Correlations and plots describe associations, not causation or medical risk.")
print("- Model evaluation and cross-validation are required beyond EDA.")
